# Apply DonSheet formatting to an existing DonSheet

This notebook formats a copy of an existing, filled DonSheet. It preserves values, formulas, sheet names, and both standard and modern Excel dropdown validations. Update `INPUT_FILE` in the next cell, then run all cells.

In [5]:
from pathlib import Path

# Change this to the filled DonSheet that should receive the formatting.
INPUT_FILE = Path("/Users/oliverglanz/Desktop/DonSheet_202821-Spring_v1.0.xlsx")
OUTPUT_FILE = INPUT_FILE.with_name(f"{INPUT_FILE.stem}_formatted.xlsx")
DROPDOWN_SOURCE_FILE = Path("/Users/oliverglanz/Documents/GitHub/SDATS_SmartSeminary/0_source_files/0000_BuildingFiles/source_DropDownMenus_v20261011.xlsx")

# The output workbook is protected with this password.
SHEET_PASSWORD = "neraK"
MIN_FORMAT_ROWS = 300

if not INPUT_FILE.is_file():
    raise FileNotFoundError(f"Input DonSheet not found: {INPUT_FILE}")
if not DROPDOWN_SOURCE_FILE.is_file():
    raise FileNotFoundError(f"Dropdown source file not found: {DROPDOWN_SOURCE_FILE}")

print(f"Input:  {INPUT_FILE}")
print(f"Output: {OUTPUT_FILE}")
print(f"Dropdown source: {DROPDOWN_SOURCE_FILE}")

Input:  /Users/oliverglanz/Desktop/DonSheet_202821-Spring_v1.0.xlsx
Output: /Users/oliverglanz/Desktop/DonSheet_202821-Spring_v1.0_formatted.xlsx
Dropdown source: /Users/oliverglanz/Documents/GitHub/SDATS_SmartSeminary/0_source_files/0000_BuildingFiles/source_DropDownMenus_v20261011.xlsx


In [6]:
from copy import copy
import re
import zipfile
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import column_index_from_string, get_column_letter
from openpyxl.utils.cell import quote_sheetname
from openpyxl.worksheet.datavalidation import DataValidation

SKIP_SHEETS = {"DropDownMenu", "InstructorMap"}
DATE_COLUMN_HEADERS = {
    "Semester Start Date {Soaterm Start Date}",
    "Pre-work Start Date {not in Banner}",
    "Pre-work End Date {not in Banner}",
    "Intensive Period Start Date {Meet Start Date}",
    "Intensive Period End Date {Meet End Date}",
    "Post-work Start Date {not in Banner}",
    "Post-work End Date {not in Banner}",
    "Semester End Date {Soaterm End Date}",
}
LOCKED_FORMULA_HEADERS = {
    "pre-work # of weeks {not in Banner}",
    "pre-work hours/period {not in Banner}",
    "intensive # of weeks {not in Banner}",
    "intensive hours/period {not in Banner}",
    "post-work # of weeks {not in Banner}",
    "post-work hours/week {not in Banner}",
    "post-work hours/period {not in Banner}",
    "sum of weeks (for check) {not in Banner}",
    "Fiscal Year {not in Banner}",
    "Semester {not in Banner}",
    "total costs {not in Banner}",
}
BLUE_HEADERS = {
    "pre-work # of weeks {not in Banner}", "pre-work hours/week {not in Banner}",
    "pre-work hours/period {not in Banner}", "intensive # of weeks {not in Banner}",
    "intensive hours/week {not in Banner}", "intensive hours/period {not in Banner}",
    "post-work # of weeks {not in Banner}", "post-work hours/week {not in Banner}",
    "post-work hours/period {not in Banner}", "sum of weeks (for check) {not in Banner}",
    "total contract hours {not in Banner}", "Year {not in Banner}", "Semester {not in Banner}",
    "Remote Employee {not in Banner}", "% Responsibility, load/contract {not in Banner}",
    "Reason for Contract {not in Banner}", "costs per credit {not in Banner}",
    "total costs {not in Banner}", "dept budget {not in Banner}",
    "account to be charged {not in Banner}", "SEM Department {Scacrse Dept}",
}
FIXED_WIDTHS = {
    "Notes {not in Banner}": 40, "Program {not in Banner}": 12, "Catalog Title": 25,
    "Section Title": 25, "Schedule Type {not in Banner}": 15,
    "Instruction Method {Inst Method}": 10, "Meeting Type": 5,
    "Semester Start Date {Soaterm Start Date}": 10, "Pre-work Start Date {not in Banner}": 10,
    "Pre-work End Date {not in Banner}": 10, "Intensive Period Start Date {Meet Start Date}": 10,
    "Intensive Period End Date {Meet End Date}": 10, "Post-work Start Date {not in Banner}": 10,
    "Post-work End Date {not in Banner}": 10, "Semester End Date {Soaterm End Date}": 10,
    "Course Beginning Time {Meet Beg Time}": 5, "Course Ending Time {Meet End Time}": 5,
    "Year {not in Banner}": 5, "Semester {not in Banner}": 7, "Room {Meet Room}": 6,
    "Building {Meet Bldg}": 30, "Instructor Name {Instr Name}": 20,
    "Instructor Email {Instr Email}": 20, "Instructor ID {Instr ID}": 10,
    "load/contract {not in Banner}": 12, "Reason for Contract {not in Banner}": 20,
    "costs per credit {not in Banner}": 5, "account to be charged {not in Banner}": 25,
    "SEM Department {Scacrse Dept}": 5, "dean email": 20, "VP finance email": 20, "HR email": 20,
}
GROUP_DEFS = [("Q", "AB"), ("AC", "AW"), ("AX", "BJ"), ("BK", "BV"), ("BW", "CE"), ("CF", "DQ")]
thin_grey_border = Border(
    left=Side(style="thin", color="D3D3D3"), right=Side(style="thin", color="D3D3D3"),
    top=Side(style="thin", color="D3D3D3"), bottom=Side(style="thin", color="D3D3D3"),
)

def format_sheet(ws):
    headers = [cell.value for cell in ws[1]]
    # A filled DonSheet needs styling only through its last populated row.
    # Dropdown validations still extend through row 5000 for future entries.
    last_row = max(ws.max_row, 2)
    # Ignore empty far-right columns left behind by older workbook operations.
    last_column = max((index for index, header in enumerate(headers, start=1) if header is not None and str(header).strip()), default=1)
    headers = headers[:last_column]

    notes_index = headers.index("Notes {not in Banner}") + 1 if "Notes {not in Banner}" in headers else 0
    ws.freeze_panes = f"{get_column_letter(notes_index + 1)}2" if notes_index else "A2"
    ws.auto_filter.ref = f"A1:{get_column_letter(last_column)}{last_row}"

    for cell in ws[1]:
        if cell.value is None or not str(cell.value).strip():
            continue
        cell.font = Font(name="Arial", size=10, bold=True, color="FF0000" if cell.value == "Notes {not in Banner}" else "000000")
        cell.fill = PatternFill(start_color="D3D3D3", end_color="D3D3D3", fill_type="solid")
        cell.alignment = Alignment(horizontal="center", vertical="bottom", wrap_text=True, text_rotation=90)
    ws.row_dimensions[1].height = 150

    start_header, end_header = "Don {not in Banner}", "SEM Department {Scacrse Dept}"
    if start_header in headers and end_header in headers:
        for column in range(headers.index(start_header) + 1, headers.index(end_header) + 2):
            ws.cell(1, column).fill = PatternFill(start_color="DDEBF7", end_color="DDEBF7", fill_type="solid")

    for column, header in enumerate(headers, start=1):
        if header in BLUE_HEADERS:
            ws.cell(1, column).fill = PatternFill(start_color="00B0F0", end_color="00B0F0", fill_type="solid")
        if header in LOCKED_FORMULA_HEADERS:
            ws.cell(1, column).fill = PatternFill(start_color="7A81FF", end_color="7A81FF", fill_type="solid")

    locked_columns = {index for index, header in enumerate(headers, start=1) if header in LOCKED_FORMULA_HEADERS}
    body_font_10 = Font(name="Arial", size=10)
    body_font_12 = Font(name="Arial", size=12)
    locked_protection = copy(ws.cell(1, 1).protection)
    locked_protection.locked = True
    unlocked_protection = copy(ws.cell(1, 1).protection)
    unlocked_protection.locked = False
    # Apply border, font, and protection in one pass over real data cells only.
    for row in ws.iter_rows(min_row=2, max_row=last_row, min_col=1, max_col=last_column):
        for cell in row:
            cell.border = thin_grey_border
            cell.font = body_font_12 if cell.column <= 7 else body_font_10
            cell.protection = locked_protection if cell.column in locked_columns else unlocked_protection

    for column, header in enumerate(headers, start=1):
        if header in DATE_COLUMN_HEADERS:
            for row in range(2, last_row + 1):
                ws.cell(row, column).number_format = "DD-MMM-YY"

    for column in range(1, last_column + 1):
        max_length = max((len(str(ws.cell(row, column).value)) for row in range(2, last_row + 1) if ws.cell(row, column).value is not None), default=0)
        ws.column_dimensions[get_column_letter(column)].width = max(4, min(max_length + 2, 60))
    for header, width in FIXED_WIDTHS.items():
        if header in headers:
            ws.column_dimensions[get_column_letter(headers.index(header) + 1)].width = width

    for dimension in ws.column_dimensions.values():
        dimension.outline_level = 0
        dimension.hidden = False
        dimension.collapsed = False
    ws.sheet_view.showOutlineSymbols = True
    ws.sheet_properties.outlinePr.summaryRight = False
    for start, end in GROUP_DEFS:
        for column in range(column_index_from_string(start) + 1, min(column_index_from_string(end), last_column) + 1):
            ws.column_dimensions[get_column_letter(column)].outline_level = 1

    ws.protection.sheet = True
    ws.protection.autoFilter = False
    ws.protection.sort = False
    ws.protection.formatColumns = False
    ws.protection.formatCells = False
    ws.protection.insertRows = False
    ws.protection.deleteRows = False
    ws.protection.set_password(SHEET_PASSWORD)

def rebuild_dropdown_menu(workbook, source_path):
    source_workbook = load_workbook(source_path, data_only=True, read_only=True)
    source_sheet = source_workbook.active
    # Iterate once in row order. Random cell access is extremely slow for
    # an openpyxl read-only worksheet.
    source_rows = source_sheet.iter_rows(values_only=True)
    source_headers = next(source_rows, ())
    dropdown_values = {
        str(header): []
        for header in source_headers
        if header is not None and str(header).strip()
    }
    seen_values = {header: set() for header in dropdown_values}
    for source_row in source_rows:
        for column, value in enumerate(source_row):
            if column >= len(source_headers):
                continue
            header = source_headers[column]
            if header is None or not str(header).strip() or value is None:
                continue
            header = str(header)
            value = str(value).strip()
            if value and value not in seen_values[header]:
                seen_values[header].add(value)
                dropdown_values[header].append(value)
    dropdown_values = {header: values for header, values in dropdown_values.items() if values}
    source_workbook.close()

    if "DropDownMenu" in workbook.sheetnames:
        workbook.remove(workbook["DropDownMenu"])
    dropdown_sheet = workbook.create_sheet("DropDownMenu")
    dropdown_ranges = {}
    for column, (header, values) in enumerate(dropdown_values.items(), start=1):
        dropdown_sheet.cell(1, column, header)
        for row, value in enumerate(values, start=2):
            dropdown_sheet.cell(row, column, value)
        letter = get_column_letter(column)
        dropdown_ranges[header] = f"{quote_sheetname('DropDownMenu')}!${letter}$2:${letter}${len(values) + 1}"
    dropdown_sheet.sheet_state = "hidden"
    return dropdown_ranges

def reapply_dropdown_validations(ws, dropdown_ranges):
    ws.data_validations.dataValidation = []
    headers = [cell.value for cell in ws[1]]
    for column, header in enumerate(headers, start=1):
        formula_range = dropdown_ranges.get(header)
        if formula_range is None:
            continue
        validation = DataValidation(type="list", formula1=f"={formula_range}", allow_blank=True)
        validation.showDropDown = False
        ws.add_data_validation(validation)
        validation.add(f"${get_column_letter(column)}$2:${get_column_letter(column)}$5000")

def capture_worksheet_extensions(workbook_path):
    # openpyxl does not retain newer x14 dropdown-validation extensions.
    # Keep those worksheet extensions and restore them after saving.
    extensions = {}
    with zipfile.ZipFile(workbook_path) as archive:
        for part_name in archive.namelist():
            if not re.fullmatch(r"xl/worksheets/sheet\d+\.xml", part_name):
                continue
            xml = archive.read(part_name).decode("utf-8")
            extension = re.search(r"<extLst>.*?</extLst>", xml, flags=re.DOTALL)
            root = re.search(r"<worksheet\b[^>]*>", xml)
            # These validations are rebuilt below as normal Excel list validations.
            # Do not restore old x14 validation records alongside the refreshed lists.
            if extension and "x14:dataValidations" in extension.group(0):
                continue
            if extension and root:
                namespaces = re.findall(r"\s(xmlns:[^=]+)=\"([^\"]+)\"", root.group(0))
                extensions[part_name] = (extension.group(0), namespaces)
    return extensions

def restore_worksheet_extensions(workbook_path, extensions):
    if not extensions:
        return
    temporary_path = workbook_path.with_name(workbook_path.stem + "_extensions_temp.xlsx")
    with zipfile.ZipFile(workbook_path) as source, zipfile.ZipFile(temporary_path, "w", zipfile.ZIP_DEFLATED) as target:
        for entry in source.infolist():
            data = source.read(entry.filename)
            if entry.filename in extensions:
                xml = data.decode("utf-8")
                extension, namespaces = extensions[entry.filename]
                xml = re.sub(r"<extLst>.*?</extLst>", "", xml, flags=re.DOTALL)
                root = re.search(r"<worksheet\b[^>]*>", xml)
                if root:
                    root_tag = root.group(0)
                    for attribute, namespace in namespaces:
                        if attribute not in root_tag:
                            root_tag = root_tag[:-1] + f' {attribute}=\"{namespace}\">'
                    xml = xml[:root.start()] + root_tag + xml[root.end():]
                xml = xml.replace("</worksheet>", extension + "</worksheet>")
                data = xml.encode("utf-8")
            target.writestr(entry, data)
    temporary_path.replace(workbook_path)

worksheet_extensions = capture_worksheet_extensions(INPUT_FILE)
wb = load_workbook(INPUT_FILE)
dropdown_ranges = rebuild_dropdown_menu(wb, DROPDOWN_SOURCE_FILE)
formatted_sheets = []
for ws in wb.worksheets:
    if ws.title in SKIP_SHEETS or ws.title.startswith("_"):
        continue
    reapply_dropdown_validations(ws, dropdown_ranges)
    format_sheet(ws)
    formatted_sheets.append(ws.title)

wb.save(OUTPUT_FILE)
wb.close()
restore_worksheet_extensions(OUTPUT_FILE, worksheet_extensions)
print(f"Formatted {len(formatted_sheets)} sheet(s). Saved: {OUTPUT_FILE}")

Formatted 11 sheet(s). Saved: /Users/oliverglanz/Desktop/DonSheet_202821-Spring_v1.0_formatted.xlsx
